# Parquet 入库查看与特征可视化（Jupyter版）

这个 Notebook 用于替代 Streamlit 版本，支持：

- 选择 `warehouse/feature_ready/V0` 下的 Parquet 文件
- 查看关键统计信息（行列数、特征数、qflag 数、时间范围、缺失率 TopN）
- 多选特征并指定时间范围绘制曲线
- 选择聚合方式（15分钟 / 日均 / 日最大 / 日最小）
- 导出当前筛选结果为 CSV

> 运行方式：按单元格从上到下执行。

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple
import warnings

import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except Exception as e:
    raise RuntimeError(
        "未检测到 ipywidgets，请先安装：pip install ipywidgets"
    ) from e

try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DEFAULT_DATA_DIR = PROJECT_ROOT / "warehouse" / "feature_ready" / "V0"
KEY_COLS = {"timestamp", "trade_date", "hh_index"}


def configure_chinese_font() -> str:
    installed = {f.name for f in font_manager.fontManager.ttflist}
    candidates = [
        "Arial Unicode MS",
        "PingFang HK",
        "Hiragino Sans GB",
        "Songti SC",
        "Heiti TC",
        "PingFang SC",
    ]
    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = [name]
            plt.rcParams["font.sans-serif"] = [name, "DejaVu Sans"]
            plt.rcParams["axes.unicode_minus"] = False
            return name
    plt.rcParams["axes.unicode_minus"] = False
    return ""


selected_font = configure_chinese_font()
warnings.filterwarnings("ignore", message=r"Glyph .* missing from current font")

if selected_font:
    print(f"Matplotlib 已启用中文字体: {selected_font}")
else:
    print("Matplotlib 未找到可用中文字体，已忽略 Glyph 告警（中文可能显示为方块）")

print(f"当前绘图后端: {'Plotly' if PLOTLY_AVAILABLE else 'Matplotlib'}")


已启用中文字体: Arial Unicode MS


In [2]:
def list_parquet_files(data_dir: Path) -> List[Path]:
    if not data_dir.exists():
        return []
    return sorted(data_dir.glob("*.parquet"))


def load_parquet(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    return df


def split_feature_columns(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    qflag_cols = [c for c in df.columns if str(c).endswith("_qflag")]
    value_cols = [c for c in df.columns if c not in KEY_COLS and c not in qflag_cols]
    return value_cols, qflag_cols


def build_missing_stats(df: pd.DataFrame, value_cols: List[str]) -> pd.DataFrame:
    if not value_cols:
        return pd.DataFrame(columns=["feature", "missing_count", "missing_ratio", "non_null_count"])
    total = len(df)
    rows = []
    for c in value_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        miss = int(s.isna().sum())
        rows.append(
            {
                "feature": c,
                "missing_count": miss,
                "missing_ratio": (miss / total) if total else 0.0,
                "non_null_count": int(total - miss),
            }
        )
    out = pd.DataFrame(rows)
    return out.sort_values(["missing_ratio", "feature"], ascending=[False, True]).reset_index(drop=True)


def aggregate_for_plot(sub: pd.DataFrame, selected_features: List[str], agg_mode: str) -> pd.DataFrame:
    if agg_mode == "原始15分钟":
        return sub[["timestamp"] + selected_features].copy()
    sub = sub.copy()
    sub["trade_date"] = pd.to_datetime(sub["timestamp"], errors="coerce").dt.normalize()
    agg_func = {
        "按日均值": "mean",
        "按日最大值": "max",
        "按日最小值": "min",
    }[agg_mode]
    grp = sub.groupby("trade_date")[selected_features].agg(agg_func).reset_index()
    grp = grp.rename(columns={"trade_date": "timestamp"})
    return grp


In [3]:
parquet_files = list_parquet_files(DEFAULT_DATA_DIR)
if not parquet_files:
    raise FileNotFoundError(f"未找到 Parquet 文件目录或目录为空: {DEFAULT_DATA_DIR}")

file_dropdown = widgets.Dropdown(
    options=[str(p) for p in parquet_files],
    value=str(parquet_files[0]),
    description="Parquet:",
    layout=widgets.Layout(width="95%")
)
agg_dropdown = widgets.Dropdown(
    options=["原始15分钟", "按日均值", "按日最大值", "按日最小值"],
    value="原始15分钟",
    description="聚合:"
)
plot_mode_toggle = widgets.ToggleButtons(
    options=["同图多曲线", "按特征分图"],
    description="绘图:"
)
feature_select = widgets.SelectMultiple(
    options=[],
    description="特征:",
    rows=12,
    layout=widgets.Layout(width="95%")
)
start_text = widgets.Text(description="开始:", placeholder="YYYY-MM-DD HH:MM")
end_text = widgets.Text(description="结束:", placeholder="YYYY-MM-DD HH:MM")
refresh_btn = widgets.Button(description="加载文件", button_style="primary")
plot_btn = widgets.Button(description="绘制曲线", button_style="success")
export_btn = widgets.Button(description="导出筛选CSV")

summary_out = widgets.Output()
plot_out = widgets.Output()
table_out = widgets.Output()

state = {"df": None, "value_cols": [], "qflag_cols": []}


def on_refresh(_=None):
    with summary_out:
        clear_output(wait=True)
        df = load_parquet(Path(file_dropdown.value))
        if df.empty:
            print("文件为空，无法展示")
            return
        if "timestamp" not in df.columns:
            print("缺少 timestamp 列，无法进行时序可视化")
            return

        value_cols, qflag_cols = split_feature_columns(df)
        state["df"] = df
        state["value_cols"] = value_cols
        state["qflag_cols"] = qflag_cols

        ts_min = df["timestamp"].min()
        ts_max = df["timestamp"].max()
        start_text.value = "" if pd.isna(ts_min) else ts_min.strftime("%Y-%m-%d %H:%M")
        end_text.value = "" if pd.isna(ts_max) else ts_max.strftime("%Y-%m-%d %H:%M")

        feature_select.options = value_cols
        feature_select.value = tuple(value_cols[: min(3, len(value_cols))])

        print(f"总行数: {len(df):,}")
        print(f"总列数: {len(df.columns):,}")
        print(f"特征列: {len(value_cols):,}")
        print(f"QFlag列: {len(qflag_cols):,}")
        print(f"时间范围: {ts_min}  ->  {ts_max}")

        miss_df = build_missing_stats(df, value_cols)
        if len(miss_df):
            display(miss_df.head(20))


def on_plot(_=None):
    with plot_out:
        clear_output(wait=True)
        df = state.get("df")
        if df is None:
            print("请先点击 加载文件")
            return

        selected_features = list(feature_select.value)
        if not selected_features:
            print("请至少选择一个特征")
            return

        start_ts = pd.to_datetime(start_text.value, errors="coerce")
        end_ts = pd.to_datetime(end_text.value, errors="coerce")
        if pd.isna(start_ts) or pd.isna(end_ts):
            print("开始/结束时间格式错误，请使用 YYYY-MM-DD HH:MM")
            return
        if end_ts < start_ts:
            print("结束时间不能早于开始时间")
            return

        sub = df[(df["timestamp"] >= start_ts) & (df["timestamp"] <= end_ts)].copy()
        if sub.empty:
            print("该时间范围无数据")
            return

        plot_df = aggregate_for_plot(sub, selected_features, agg_dropdown.value)

        if PLOTLY_AVAILABLE:
            if plot_mode_toggle.value == "同图多曲线":
                long_df = plot_df.melt(
                    id_vars=["timestamp"],
                    value_vars=selected_features,
                    var_name="feature",
                    value_name="value",
                )
                fig = px.line(
                    long_df,
                    x="timestamp",
                    y="value",
                    color="feature",
                    title="特征曲线",
                )
                fig.update_layout(height=460)
                fig.show()
            else:
                long_df = plot_df.melt(
                    id_vars=["timestamp"],
                    value_vars=selected_features,
                    var_name="feature",
                    value_name="value",
                )
                fig = px.line(
                    long_df,
                    x="timestamp",
                    y="value",
                    facet_row="feature",
                    title="按特征分图",
                    height=max(280 * len(selected_features), 360),
                )
                fig.update_layout(showlegend=False)
                fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
                fig.show()
        else:
            if plot_mode_toggle.value == "同图多曲线":
                ax = plot_df.set_index("timestamp")[selected_features].plot(figsize=(14, 5))
                ax.set_title("特征曲线")
                ax.set_xlabel("timestamp")
                ax.set_ylabel("value")
                plt.tight_layout()
                plt.show()
            else:
                n = len(selected_features)
                fig, axes = plt.subplots(n, 1, figsize=(14, max(3 * n, 4)), sharex=True)
                if n == 1:
                    axes = [axes]
                for i, feat in enumerate(selected_features):
                    axes[i].plot(plot_df["timestamp"], plot_df[feat])
                    axes[i].set_title(feat)
                    axes[i].set_ylabel("value")
                axes[-1].set_xlabel("timestamp")
                plt.tight_layout()
                plt.show()

    with table_out:
        clear_output(wait=True)
        preview_cols = ["timestamp"] + selected_features
        display(sub[preview_cols].head(200))


def on_export(_=None):
    df = state.get("df")
    if df is None:
        print("请先点击 加载文件")
        return

    selected_features = list(feature_select.value)
    if not selected_features:
        print("请至少选择一个特征")
        return

    start_ts = pd.to_datetime(start_text.value, errors="coerce")
    end_ts = pd.to_datetime(end_text.value, errors="coerce")
    sub = df[(df["timestamp"] >= start_ts) & (df["timestamp"] <= end_ts)].copy()

    preview_cols = ["timestamp"] + selected_features

    export_df = sub[preview_cols].copy()
    out_path = PROJECT_ROOT / "warehouse" / "feature_ready" / "V0" / "feature_selection_export.csv"
    export_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"导出完成: {out_path}")


refresh_btn.on_click(on_refresh)
plot_btn.on_click(on_plot)
export_btn.on_click(on_export)


In [4]:
ui = widgets.VBox([
    widgets.HTML("<h3>Parquet 入库查看与特征可视化（Jupyter版）</h3>"),
    file_dropdown,
    widgets.HBox([agg_dropdown, plot_mode_toggle]),
    feature_select,
    widgets.HBox([start_text, end_text]),
    widgets.HBox([refresh_btn, plot_btn, export_btn]),
    widgets.HTML("<b>关键统计 / 缺失率 Top20</b>"),
    summary_out,
    widgets.HTML("<b>曲线图</b>"),
    plot_out,
    widgets.HTML("<b>筛选预览（前200行）</b>"),
    table_out,
])

display(ui)
on_refresh()
